In [17]:
# ============================================================
# ⚙️  CONFIGURAZIONE MISCELE  — modifica solo questo dizionario
# ============================================================
# Per ogni miscela definisci:
#   root_file        → nome del file ROOT prodotto dal notebook di fit
#   scaling_threshold→ V sopra cui applicare scaling ×10 (None = nessuno scaling)
#   fit_min_voltage  → tensione minima inclusa nel fit (None = usa tutti i punti)

MIXTURE_CONFIG = {
    "Ar_CO2_Iso_93_5_2": {
        "root_file": "output_Ar_CO2_Iso_93_5_2_MM2DNAP1.root",
        "scaling_threshold": 540,   # V > 540 → ×10
        "fit_min_voltage":   None,  # usa tutti i punti
    },
    "Ar_CO2_Iso_88_10_2": {
        "root_file": "output_Ar_CO2_Iso_88_10_2_MM2DNAP1.root",
        "scaling_threshold": 595,   # V > 595 → ×10
        "fit_min_voltage":   530,   # fit parte da 530 V
    },
}


In [18]:
import os
import ROOT
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


In [19]:
def read_tree(fname):

    f = ROOT.TFile(fname)
    t = f.Get("fitResults")

    V, mu, chi2, ndf = [], [], [], []

    for e in t:
        V.append(e.V)
        mu.append(e.mu1)
        chi2.append(e.chi2)
        ndf.append(e.ndf)

    V = np.array(V)
    mu = np.array(mu)
    chi2 = np.array(chi2)
    ndf = np.array(ndf)

    chi2ndf = chi2 / np.maximum(ndf, 1)

    return V, mu, chi2ndf

In [20]:
def compute_errors(mu, chi2ndf):

    scale = np.maximum(chi2ndf, 1.0)
    err = 0.05 * mu * scale
    return err


def apply_gain_scaling(V, mu, mixture):

    SCALA_1V  = 1.0
    SCALA_10V = 10.0

    # legge la soglia direttamente dalla configurazione centralizzata
    cfg = MIXTURE_CONFIG.get(mixture, {})
    threshold = cfg.get("scaling_threshold", None)

    guadagno_scalato = []
    for v_val, g_val in zip(V, mu):
        scale_up = (threshold is not None) and (v_val > threshold)
        guadagno_scalato.append(g_val * SCALA_10V if scale_up else g_val * SCALA_1V)

    return np.array(guadagno_scalato)


def build_gain_table(V, guadagno_scalato):
    return pd.DataFrame({
        "Tensione [V]":    V.astype(int),
        "Guadagno [canali]": np.round(guadagno_scalato, 1)
    })


def ensure_output_dir(path):
    os.makedirs(path, exist_ok=True)


In [21]:
def weighted_exp_fit(V, G, sigma_G, nfit=5, min_voltage=None):

    if min_voltage is not None:
        mask = V >= min_voltage
        V_sel = V[mask]
        G_sel = G[mask]
        sigma_sel = sigma_G[mask]
    else:
        V_sel = V
        G_sel = G
        sigma_sel = sigma_G

    if len(V_sel) == 0:
        raise ValueError(f"Nessun punto disponibile per il fit con min_voltage={min_voltage}")

    # seleziona primi punti del sottoinsieme scelto
    V_fit = V_sel[:nfit]
    G_fit = G_sel[:nfit]
    sigma_fit = sigma_sel[:nfit]

    # passa a log
    logG = np.log(G_fit)

    # 🔥 propagazione errore:
    # σ_log = σ_G / G
    sigma_log = sigma_fit / G_fit

    # pesi = 1/σ²
    weights = 1.0 / (sigma_log**2)

    # fit pesato
    coeff = np.polyfit(V_fit, logG, 1, w=np.sqrt(weights))

    B = coeff[0]
    logA = coeff[1]
    A = np.exp(logA)

    return A, B, V_fit, G_fit

In [22]:
def plot_gain_with_pull(fname, mixture, save_dir="output", fit_min_voltage=None):

    V, mu, chi2ndf = read_tree(fname)

    # ordina
    order = np.argsort(V)
    V       = V[order]
    mu      = mu[order]
    chi2ndf = chi2ndf[order]

    # ------------------
    # Gestione cambio scala multicanale
    # ------------------
    guadagno_scalato = apply_gain_scaling(V, mu, mixture)

    # errori calcolati sul guadagno scalato
    err = compute_errors(guadagno_scalato, chi2ndf)

    # ✅ FIT PESATO sui dati scalati
    A, B, V_fit_used, G_fit_used = weighted_exp_fit(
        V, guadagno_scalato, err,
        nfit=min(5, len(V)),
        min_voltage=fit_min_voltage
    )

    fit_label = f"V >= {fit_min_voltage} V" if fit_min_voltage is not None else "tutti i punti"
    print(f"[FIT {mixture}] G = {A:.3e} * exp({B:.4f} V) | fit su {fit_label}")

    # modello
    model = A * np.exp(B * V)

    # pull corretto
    pull = (guadagno_scalato - model) / err

    ensure_output_dir(save_dir)
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(7, 6),
        gridspec_kw={'height_ratios': [3, 1]},
        sharex=True
    )

    # =====================
    # TOP
    # =====================
    ax1.errorbar(V, guadagno_scalato, yerr=err, fmt='s', color='blue', label='data (scaled)')

    # fit SOLIDO sui punti usati nel fit
    ax1.plot(V_fit_used, G_fit_used, color='red', linewidth=2, label='fit esponenziale')

    # linea tratteggiata estesa a tutto il range
    V_full = np.linspace(min(V), max(V), 200)
    G_full = A * np.exp(B * V_full)
    ax1.plot(V_full, G_full, '--', color='red', alpha=0.7, label='estensione del fit')

    ax1.set_yscale("log")
    ax1.set_ylabel("Posizione picco [canali]")
    ax1.set_title(mixture)
    ax1.grid(True)
    ax1.legend(loc='lower right')

    ax1.text(
        0.05, 0.85,
        f"G = {A:.2e} e^({B:.4f} V)",
        transform=ax1.transAxes
    )

    # =====================
    # PULL
    # =====================
    ax2.axhline(0,  color='black')
    ax2.axhline(+3, color='gray', linestyle='--')
    ax2.axhline(-3, color='gray', linestyle='--')

    ax2.errorbar(V, pull, fmt='o', color='black')

    ax2.set_xlabel("Tensione [V]")
    ax2.set_ylabel("Pull")
    ax2.grid(True)

    plt.tight_layout()
    image_path = os.path.join(save_dir, f"gain_{mixture}.png")
    fig.savefig(image_path, dpi=200)
    plt.close(fig)

    print(f"Saved plot: {image_path}")

    return V, guadagno_scalato, err, A, B


In [23]:
output_dir = "output"
ensure_output_dir(output_dir)

for mixture, cfg in MIXTURE_CONFIG.items():

    fname = cfg["root_file"]
    fit_min_voltage = cfg.get("fit_min_voltage", None)

    if not os.path.exists(fname):
        print(f"[SKIP] file ROOT mancante per {mixture}: {fname}")
        continue

    V, mu, chi2ndf = read_tree(fname)
    order = np.argsort(V)
    V  = V[order]
    mu = mu[order]

    guadagno_scalato = apply_gain_scaling(V, mu, mixture)
    table = build_gain_table(V, guadagno_scalato)

    gas_dir  = os.path.join(output_dir, mixture)
    ensure_output_dir(gas_dir)

    csv_path = os.path.join(gas_dir, f"table_{mixture}.csv")
    table.to_csv(csv_path, index=False)

    print(f"\n=== Tabella {mixture} ===")
    display(table)
    print(f"Saved table: {csv_path}")

    plot_gain_with_pull(fname, mixture, save_dir=gas_dir, fit_min_voltage=fit_min_voltage)



=== Tabella Ar_CO2_Iso_93_5_2 ===


,Tensione [V],Guadagno [canali]
0,490,725.6
1,500,1049.9
2,510,1482.2
3,515,1774.9
4,520,2112.4
5,525,2572.1
6,530,3129.6
7,535,3822.9
8,540,4739.7
9,545,6159.3


Saved table: output/Ar_CO2_Iso_93_5_2/table_Ar_CO2_Iso_93_5_2.csv
[FIT Ar_CO2_Iso_93_5_2] G = 2.114e-05 * exp(0.0354 V) | fit su tutti i punti
Saved plot: output/Ar_CO2_Iso_93_5_2/gain_Ar_CO2_Iso_93_5_2.png

=== Tabella Ar_CO2_Iso_88_10_2 ===


,Tensione [V],Guadagno [canali]
0,490,1336.5
1,500,838.1
2,510,481.4
3,520,1034.1
4,530,644.7
5,540,887.4
6,550,1132.1
7,560,1469.4
8,570,2081.9
9,580,2777.3


Saved table: output/Ar_CO2_Iso_88_10_2/table_Ar_CO2_Iso_88_10_2.csv
[FIT Ar_CO2_Iso_88_10_2] G = 1.987e-04 * exp(0.0283 V) | fit su V >= 530 V
Saved plot: output/Ar_CO2_Iso_88_10_2/gain_Ar_CO2_Iso_88_10_2.png


In [14]:
# Export dizionario gain_data riusabile da altri notebook
import json
from pathlib import Path

base_dir = Path(output_dir)

mixture_csv = {
    'Ar_CO2_Iso_93_5_2': base_dir / 'Ar_CO2_Iso_93_5_2' / 'table_Ar_CO2_Iso_93_5_2.csv',
    'Ar_CO2_Iso_88_10_2': base_dir / 'Ar_CO2_Iso_88_10_2' / 'table_Ar_CO2_Iso_88_10_2.csv',
}

gain_data = {}

for mixture, csv_file in mixture_csv.items():
    if not csv_file.exists():
        print(f'[{mixture}] file mancante: {csv_file}')
        gain_data[mixture] = {}
        continue

    df = pd.read_csv(csv_file)

    # Colonne attese dal notebook
    v_col = 'Tensione [V]'
    g_col = 'Guadagno [canali]'
    if v_col not in df.columns or g_col not in df.columns:
        raise KeyError(f'Colonne attese non trovate in {csv_file}. Colonne disponibili: {list(df.columns)}')

    # Dizionario {voltage: gain_value}
    mixture_dict = {
        float(v): float(g)
        for v, g in zip(df[v_col], df[g_col])
        if pd.notna(v) and pd.notna(g)
    }
    gain_data[mixture] = mixture_dict

# Salvataggio dizionari dentro la folder della singola miscela
for mixture, data in gain_data.items():
    mixture_dir = base_dir / mixture
    mixture_dir.mkdir(parents=True, exist_ok=True)

    py_path = mixture_dir / 'gain_data.py'
    json_path = mixture_dir / 'gain_data.json'

    py_lines = [
        '# Auto-generated by draw_gain.ipynb',
        '# --- Gain data (single mixture) ---',
        '# Format: {voltage: gain_value}',
        'gain_data = ' + repr({mixture: data}),
        ''
    ]
    py_path.write_text('\n'.join(py_lines), encoding='utf-8')

    json_ready = {str(v): g for v, g in data.items()}
    json_path.write_text(json.dumps(json_ready, indent=2), encoding='utf-8')

    print(f'[{mixture}] File Python: {py_path.resolve()}')
    print(f'[{mixture}] File JSON:   {json_path.resolve()}')

# Pulizia eventuali file legacy nella root output
legacy_py = base_dir / 'gain_data.py'
legacy_json = base_dir / 'gain_data.json'
if legacy_py.exists():
    legacy_py.unlink()
    print(f'Rimosso file legacy: {legacy_py.resolve()}')
if legacy_json.exists():
    legacy_json.unlink()
    print(f'Rimosso file legacy: {legacy_json.resolve()}')

print('Dizionari creati con successo per miscela:')
print(gain_data)


[Ar_CO2_Iso_93_5_2] File Python: /Users/adelinadonofrio/Desktop/MPGD/output/Ar_CO2_Iso_93_5_2/gain_data.py
[Ar_CO2_Iso_93_5_2] File JSON:   /Users/adelinadonofrio/Desktop/MPGD/output/Ar_CO2_Iso_93_5_2/gain_data.json
[Ar_CO2_Iso_88_10_2] File Python: /Users/adelinadonofrio/Desktop/MPGD/output/Ar_CO2_Iso_88_10_2/gain_data.py
[Ar_CO2_Iso_88_10_2] File JSON:   /Users/adelinadonofrio/Desktop/MPGD/output/Ar_CO2_Iso_88_10_2/gain_data.json
Dizionari creati con successo per miscela:
{'Ar_CO2_Iso_93_5_2': {490.0: 725.6, 500.0: 1049.9, 510.0: 1482.2, 515.0: 1774.9, 520.0: 2112.4, 525.0: 2572.1, 530.0: 3129.6, 535.0: 3822.9, 540.0: 4739.7, 545.0: 6159.3, 550.0: 7902.1}, 'Ar_CO2_Iso_88_10_2': {490.0: 1336.5, 500.0: 838.1, 510.0: 481.4, 520.0: 1034.1, 530.0: 644.7, 540.0: 887.4, 550.0: 1132.1, 560.0: 1469.4, 570.0: 2081.9, 580.0: 2777.3, 590.0: 3887.1, 600.0: 5241.6}}
